In [ ]:
from contextlib import nullcontext
from pathlib import Path
import os
import sys

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from sam2.build_sam import build_sam2_video_predictor

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.sam2.inference import (
    init_frame_state,
    add_box_prompts,
    propagate_inference,
)
from src.sam2.utils import retrieve_prompts_from_inference_state
from src.common.schemas import Box2D
from src.common.visualize.segmentation import plot_instance_mask_with_prompt, plot_instance_mask
from src.common.visualize.colors import TABLEAU10_NAMES

# Resolve paths relative to this notebook directory
ROOT = Path.cwd().parent / "sam2"
CHECKPOINT = ROOT / "checkpoints/sam2.1_hiera_large.pt"
MODEL_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
VIDEO_DIR = ROOT / "notebooks/videos/bedroom"
device = "cuda" if torch.cuda.is_available() else "cpu"
autocast_context = (
    torch.autocast("cuda", dtype=torch.bfloat16)
    if device == "cuda"
    else nullcontext()
)
color_dict = {i: TABLEAU10_NAMES[i] for i in range(10)}

# Build the SAM2 model and create a predictor
predictor = build_sam2_video_predictor(str(MODEL_CONFIG), str(CHECKPOINT), device=device)
print("Model loaded")

# Load the frame images from the video directory
frame_names = [
    p for p in os.listdir(VIDEO_DIR)
    if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
]
frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))
frames = [Image.open(VIDEO_DIR / p) for p in frame_names]

# Get the inferenece state
inference_state = init_frame_state(predictor, frames)
image_height, image_width = inference_state["video_height"], inference_state["video_width"]

In [ ]:
# Inference with Bounding Boxes Prompt
predictor.reset_state(inference_state)
# Box prompt
box_prompts = [
    Box2D(xyxy=np.array([140, 120, 290, 420], dtype=np.float32), track_id=0),
    Box2D(xyxy=np.array([300, 0, 500, 400], dtype=np.float32), track_id=3),
]
# Add the box prompts to the first frame and get the predicted masks for the boxes
predicted_instances = add_box_prompts(
    predictor=predictor,
    inference_state=inference_state,
    frame_idx=0,
    box_prompts=box_prompts,
)

# Plot the image
first_frame = frames[0]
plt.imshow(first_frame)
plt.axis("off")
# Show the box prompt and the predicted masks on the first frame
for i in range(len(predicted_instances)):
    obj_id = predicted_instances[i].box.track_id
    plot_instance_mask_with_prompt(
        instance=predicted_instances[i],
        image_height=first_frame.height,
        image_width=first_frame.width,
        prompt_box=box_prompts[i],
        color=color_dict[obj_id],
        prompt_box_color=color_dict[obj_id]
    )
plt.show()

In [ ]:
# run propagation throughout the frame sequence
result_instances = propagate_inference(predictor, inference_state)
out_box_prompts, out_point_prompts = retrieve_prompts_from_inference_state(inference_state)

vis_frame_stride = 30
for out_frame_idx in range(0, len(frame_names), vis_frame_stride):
    # Plot the image
    plt.imshow(frames[out_frame_idx])
    plt.axis("off")
    plt.title(f"Frame {out_frame_idx}")
    # Plot the predicted masks for the current frame
    for obj_id, result_instance in result_instances[out_frame_idx].items():
        # Get the prompts for the current frame and obj_id
        box_prompt = next((prompt for prompt in out_box_prompts
                           if prompt["frame_idx"] == out_frame_idx and prompt["obj_id"] == obj_id), 
                          None)
        point_prompts = next((prompt for prompt in out_point_prompts
                              if prompt["frame_idx"] == out_frame_idx and prompt["obj_id"] == obj_id), 
                             None)
        if box_prompt is not None:
            box_prompt = Box2D(xyxy=box_prompt["xyxy"], track_id=obj_id)
            plot_instance_mask_with_prompt(
                instance=result_instance,
                image_height=frames[out_frame_idx].height,
                image_width=frames[out_frame_idx].width,
                prompt_box=box_prompt,
                color=color_dict[obj_id],
                prompt_box_color=color_dict[obj_id]
            )
        else:
            plot_instance_mask(
                instance=result_instance,
                image_height=frames[out_frame_idx].height,
                image_width=frames[out_frame_idx].width,
                color=color_dict[obj_id]
            )
    plt.show()